<a href="https://colab.research.google.com/github/yeonsub/Learning_Material_for_Creation/blob/main/RNN_Edu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
import torch.nn as nn
import numpy as np

class EmotionRNN(nn.Module):

    def __init__(self, input_size, hidden_size, output_size):
        super(EmotionRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True) # Changed: Added batch_first=True
        self.dense = nn.Linear(hidden_size, output_size)

    def forward(self, input_tensor, hidden_tensor):
        rnn_output, _ = self.rnn(input_tensor, hidden_tensor)
        # If batch_first=True, rnn_output shape is (batch_size, sequence_length, hidden_size)
        # Changed: Access the last hidden state of the sequence for each item in the batch, and keep batch dimension
        last_one = rnn_output[:, -1, :]
        output = self.dense(last_one)
        return output # Return raw logits for CrossEntropyLoss


In [26]:
batch_size = 1
hidden_size = 32

dataset = [
    ["I Love You", "Good"],
    ["I Hate You", "Bad"],
    ["I Like You", "Good"],
    ["You Dislike Me", "Bad"],
    ["I Like Dislike", "Calm"],
    ["I You You", "Calm"]
]
sequence_length = len(dataset)

word_to_index_map = {
    "I": 0,
    "Love": 1,
    "Hate": 2,
    "You": 3,
    "Like":4,
    "Dislike":5,
    "Me":6
}
input_size = len(word_to_index_map)

label_to_index_map = {
    "Good": 0,
    "Bad": 1,
    "Calm": 2
}
output_size = len(label_to_index_map) # Fix: Changed output_size to match the number of unique labels


In [29]:
# Helper function to get word index from word_to_index_map (assuming it's globally available)
def sentence_to_input_tensor(sentence, word_map, input_dim):
    # Filter out words not in the word_map
    known_words_indices = [word_map[word] for word in sentence.split() if word in word_map]

    if not known_words_indices:
        # Return None or an empty tensor if no known words are found
        # This signals to the calling function that no valid input can be formed
        return None, False

    one_hot_vectors = [np.eye(input_dim)[idx] for idx in known_words_indices]
    input_tensor = torch.tensor(one_hot_vectors, dtype=torch.float)
    # Reshape to (batch_size, sequence_length, input_size) as batch_first=True
    return input_tensor.reshape(1, len(known_words_indices), input_dim), True

In [40]:
n_epoch = 100

model = EmotionRNN(input_size, hidden_size, output_size)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Set model to training mode
model.train()

for epoch in range(n_epoch):
    totalloss=0
    for data in dataset:
        sentence = data[0]
        target = data[1]

        input_tensor_tuple = sentence_to_input_tensor(sentence,word_to_index_map, input_size)

        # Check if any known words were found
        if not input_tensor_tuple[1]: # if has_known_words is False
            # Skip this data point if no known words, or handle as needed
            print(f"Skipping sentence '{sentence}' due to unknown words.")
            continue

        input_tensor = input_tensor_tuple[0] # Unpack the actual tensor

        target_to_index = label_to_index_map[target]
        target_tensor = torch.tensor([target_to_index], dtype=torch.long)

        # Changed: Hidden state should also be batch_first if using batch_first=True for input (though not strictly required for initial zero state)
        hidden_state = torch.zeros(1, 1, hidden_size, dtype=torch.float)
        prediction = model(input_tensor, hidden_state)

        loss = criterion(prediction, target_tensor)

        # Zero gradients, perform a backward pass, and update the weights.
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        totalloss +=loss.item()

    # Optionally print total loss for the epoch
    print(f'Epoch {epoch+1}/{n_epoch}, Loss: {totalloss:.4f}')

Epoch 1/100, Loss: 6.7619
Epoch 2/100, Loss: 6.6181
Epoch 3/100, Loss: 6.5438
Epoch 4/100, Loss: 6.4761
Epoch 5/100, Loss: 6.4099
Epoch 6/100, Loss: 6.3435
Epoch 7/100, Loss: 6.2758
Epoch 8/100, Loss: 6.2059
Epoch 9/100, Loss: 6.1330
Epoch 10/100, Loss: 6.0565
Epoch 11/100, Loss: 5.9757
Epoch 12/100, Loss: 5.8902
Epoch 13/100, Loss: 5.7992
Epoch 14/100, Loss: 5.7025
Epoch 15/100, Loss: 5.5995
Epoch 16/100, Loss: 5.4901
Epoch 17/100, Loss: 5.3741
Epoch 18/100, Loss: 5.2516
Epoch 19/100, Loss: 5.1228
Epoch 20/100, Loss: 4.9883
Epoch 21/100, Loss: 4.8486
Epoch 22/100, Loss: 4.7046
Epoch 23/100, Loss: 4.5574
Epoch 24/100, Loss: 4.4080
Epoch 25/100, Loss: 4.2575
Epoch 26/100, Loss: 4.1071
Epoch 27/100, Loss: 3.9578
Epoch 28/100, Loss: 3.8106
Epoch 29/100, Loss: 3.6662
Epoch 30/100, Loss: 3.5252
Epoch 31/100, Loss: 3.3882
Epoch 32/100, Loss: 3.2554
Epoch 33/100, Loss: 3.1272
Epoch 34/100, Loss: 3.0036
Epoch 35/100, Loss: 2.8848
Epoch 36/100, Loss: 2.7706
Epoch 37/100, Loss: 2.6611
Epoch 38/1

### 모델 평가 루틴

학습된 `EmotionRNN` 모델을 사용하여 새로운 문장의 감성을 예측하고, 모델의 정확도를 평가하는 루틴을 작성합니다.

In [41]:
def evaluate_model(model, sentence, word_to_index, index_to_label, input_size, hidden_size):
    model.eval() # Set the model to evaluation mode
    with torch.no_grad(): # Disable gradient calculations
        input_tensor, has_known_words = sentence_to_input_tensor(sentence, word_to_index, input_size)

        if not has_known_words:
            return "알 수 없는 단어 포함", None # Return a specific message for unknown words

        # Changed: Hidden state should also be batch_first if using batch_first=True for input (though not strictly required for initial zero state)
        hidden_state = torch.zeros(1, 1, hidden_size, dtype=torch.float)
        prediction = model(input_tensor, hidden_state)

        # Get the predicted class index
        predicted_index = torch.argmax(prediction, dim=1).item() # prediction is now raw logits, argmax still works
        predicted_label = index_to_label[predicted_index]
        return predicted_label, prediction

# Create a reverse mapping for labels to display results
index_to_label_map = {v: k for k, v in label_to_index_map.items()}

print("\n--- Model Evaluation ---")
correct_predictions = 0
total_samples = len(dataset)

for sentence, true_label in dataset:
    predicted_label, _ = evaluate_model(model, sentence, word_to_index_map, index_to_label_map, input_size, hidden_size)
    print(f'Sentence: "{sentence}" | True Label: {true_label} | Predicted Label: {predicted_label}')
    if predicted_label == true_label:
        correct_predictions += 1

accuracy = (correct_predictions / total_samples) * 100
print(f"\nAccuracy on the training dataset: {accuracy:.2f}%")


--- Model Evaluation ---
Sentence: "I Love You" | True Label: Good | Predicted Label: Good
Sentence: "I Hate You" | True Label: Bad | Predicted Label: Bad
Sentence: "I Like You" | True Label: Good | Predicted Label: Good
Sentence: "You Dislike Me" | True Label: Bad | Predicted Label: Bad
Sentence: "I Like Dislike" | True Label: Calm | Predicted Label: Calm
Sentence: "I You You" | True Label: Calm | Predicted Label: Calm

Accuracy on the training dataset: 100.00%


### 새로운 문장으로 감정 예측하기

In [44]:
print("\n--- 새로운 문장 감정 예측 ---")
user_sentence = input("감정을 예측할 문장을 입력하세요: ")

predicted_label, _ = evaluate_model(model, user_sentence, word_to_index_map, index_to_label_map, input_size, hidden_size)

if predicted_label == "알 수 없는 단어 포함":
    print(f'입력 문장: "{user_sentence}" | 예측 실패: 문장에 모델이 학습한 단어가 없습니다. (알 수 없는 단어 포함)')
else:
    print(f'입력 문장: "{user_sentence}" | 예측된 감정: {predicted_label}')


--- 새로운 문장 감정 예측 ---
감정을 예측할 문장을 입력하세요: I I I
입력 문장: "I I I" | 예측된 감정: Calm
